# TFM · Comparativa de modelos de PLN para clasificar reclamos (CFPB)
**Autora:** Stefanny A. Solari Camargo · Máster Universitario en IA (UNIR)

Este cuaderno ejecuta TODO el experimento de los capítulos 4 y 5 del TFM:
Naive Bayes y SVM lineal (TF-IDF) frente a RoBERTa-base (fine-tuning), con tres semillas,
F1 macro, coste computacional, análisis por longitud, contraste de McNemar (Holm) y
un experimento temporal complementario.

**Cómo usarlo en Google Colab**
1. Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
2. Ejecutar las celdas en orden (Entorno → Ejecutar todo).
3. Al final se genera `resultados/` (CSV, figuras y `tablas_capitulo5.md`). Descárgalo y
   copia los números en las Tablas 7–12 del documento Word.

Si solo quieres probar que todo funciona, pon `QUICK_TEST = True` en la celda de configuración.

In [1]:
!!pip install -q transformers scikit-learn pandas nltk matplotlib

[]

In [2]:
import os, re, json, time, random, platform, hashlib, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix)
warnings.filterwarnings("ignore")

## 0. Configuración (único bloque que conviene editar)

In [3]:
CONFIG = dict(
    DATA_PATH      = None,                   # (ya no se usa: los datos vienen del archivo FOIA, ver 0b)
    START_YEAR     = 2020,                   # periodo de estudio: [START_YEAR, END_YEAR]
    END_YEAR       = 2025,
    N_SAMPLE       = 60_000,                 # narrativas de trabajo (muestra aleatoria, distribución natural)
    SEEDS          = [42, 43, 44],           # tres repeticiones (partición + entrenamiento)
    SPLIT          = (0.70, 0.10, 0.20),     # entrenamiento / validación / prueba (estratificado)
    MIN_WORDS      = 10,                     # se descartan narrativas con menos palabras
    # --- Transformer ---
    MODEL_NAME     = "roberta-base",
    MAX_LEN        = 256,                    # incluye tokens especiales
    HEAD_TOKENS    = 64,                     # truncamiento inicio+final (Sun et al., 2019): 25 % / 75 %
    LR             = 2e-5,
    EPOCHS         = 3,
    BATCH          = 32,
    WARMUP         = 0.06,
    WEIGHT_DECAY   = 0.01,
    # --- Clásicos: rejillas evaluadas en validación ---
    NB_ALPHAS      = [0.01, 0.1, 0.3, 1.0],
    SVM_CS         = [0.1, 0.3, 1.0, 3.0],
    TFIDF          = dict(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True, max_features=100_000),
    CPU_LATENCY_N  = 300,                    # nº de reclamos para medir latencia del Transformer en CPU
    RUN_TEMPORAL   = True,                   # experimento complementario: entrenar en años previos, probar en el último
    SKIP_TRANSFORMER = False,                # True = solo modelos clásicos (prueba rápida sin GPU)
    OUT_DIR        = "resultados",
)
QUICK_TEST = False
if QUICK_TEST:
    CONFIG.update(N_SAMPLE=4000, SEEDS=[42], EPOCHS=1, RUN_TEMPORAL=False)
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception:
        pass

## 0b. Obtener los datos de la CFPB (con la columna de narrativa)
**Importante:** desde el 14-ago-2026 la CFPB dejó de publicar las narrativas en `complaints.csv.zip`.
Las narrativas ya publicadas (dic-2011 a 14-ago-2026) están en el **«Narratives Archive»** de su sala de lectura FOIA,
en varios ZIP (`CCDB_Export_n_...`). Esta celda descarga los que cubren 2020–2025 (exportaciones 2 a 14),
**enseña qué contiene cada uno** y verifica que tengan la narrativa.
Fuente: https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/

In [4]:
import os, subprocess, zipfile, glob
BASE = "https://files.consumerfinance.gov/f/documents/"
EXPORTS = [  # (n, nombre) -> periodo; 2020-2025 = exportaciones 2..14 (2026 = 15..21, no se usan)
    "CCDB_Export_2_May_2018_through_April_2021.zip",
    "CCDB_Export_3_May_2021_through_October_2022.zip",
    "CCDB_Export_4_November_2022_through_August_2023.zip",
    "CCDB_Export_5_September_2023_through_March_2024.zip",
    "CCDB_Export_6_April_2024_through_July_2024.zip",
    "CCDB_Export_7_August_2024_through_October_2024.zip",
    "CCDB_Export_8_November_2024_through_December_2024.zip",
    "CCDB_Export_9_January_2025_through_February_2025.zip",
    "CCDB_Export_10_March_2025_through_April_2025.zip",
    "CCDB_Export_11_May_2025_through_June_2025.zip",
    "CCDB_Export_12_July_2025_through_August_2025.zip",
    "CCDB_Export_13_September_2025_through_October_2025.zip",
    "CCDB_Export_14_November_2025_through_December_2025.zip",
]
DATA_DIR = "ccdb_archivo"
os.makedirs(DATA_DIR, exist_ok=True)
for name in EXPORTS:
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        continue
    print("Descargando", name, "...")
    r = subprocess.run(["wget", "-q", "-O", path, BASE + name])
    if r.returncode != 0 or not os.path.exists(path) or os.path.getsize(path) < 100_000:
        if os.path.exists(path): os.remove(path)
        print("   !! no se pudo descargar", name)

DATA_FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.zip")), key=lambda p: int(re.search(r"Export_(\d+)_", p).group(1)))
if not DATA_FILES:
    raise FileNotFoundError("No se descargó ningún ZIP del archivo de narrativas. Copia y pégame el mensaje de error.")

# ---- Inspección: qué hay dentro de cada ZIP (formato y columnas) ----
for p in DATA_FILES[:2]:
    with zipfile.ZipFile(p) as z:
        print("\n", os.path.basename(p), f"({os.path.getsize(p)/1e6:.0f} MB)")
        for i in z.infolist()[:5]:
            print("   contenido:", i.filename, f"{i.file_size/1e6:.0f} MB")
        m = [i.filename for i in z.infolist() if i.filename.lower().endswith(".csv")]
        if m:
            print("   columnas:", list(pd.read_csv(z.open(m[0]), nrows=0).columns))
print("\nZIP disponibles:", len(DATA_FILES), "de", len(EXPORTS))

Descargando CCDB_Export_2_May_2018_through_April_2021.zip ...
Descargando CCDB_Export_3_May_2021_through_October_2022.zip ...
Descargando CCDB_Export_4_November_2022_through_August_2023.zip ...
Descargando CCDB_Export_5_September_2023_through_March_2024.zip ...
Descargando CCDB_Export_6_April_2024_through_July_2024.zip ...
Descargando CCDB_Export_7_August_2024_through_October_2024.zip ...
Descargando CCDB_Export_8_November_2024_through_December_2024.zip ...
Descargando CCDB_Export_9_January_2025_through_February_2025.zip ...
Descargando CCDB_Export_10_March_2025_through_April_2025.zip ...
Descargando CCDB_Export_11_May_2025_through_June_2025.zip ...
Descargando CCDB_Export_12_July_2025_through_August_2025.zip ...
Descargando CCDB_Export_13_September_2025_through_October_2025.zip ...
Descargando CCDB_Export_14_November_2025_through_December_2025.zip ...

 CCDB_Export_2_May_2018_through_April_2021.zip (165 MB)
   contenido: CCDB_Export_2_May_2018_through_April_2021.csv 783 MB
   columnas

## 1. Carga, armonización de la variable objetivo y depuración
La CFPB ha renombrado y reagrupado sus productos a lo largo del tiempo. `PRODUCT_MAP` armoniza los
nombres a 8 clases. **Revisa la salida de `value_counts()` y completa/ajusta el diccionario** si
aparece algún producto sin mapear (se descarta y se informa de cuántos registros se pierden).

In [5]:
PRODUCT_MAP = {
    # reportes de crédito
    "credit reporting, credit repair services, or other personal consumer reports": "credit_reporting",
    "credit reporting or other personal consumer reports": "credit_reporting",
    "credit reporting": "credit_reporting",
    "credit repair": "credit_reporting",
    # cobranza
    "debt collection": "debt_collection",
    # hipotecas
    "mortgage": "mortgage",
    # tarjetas
    "credit card or prepaid card": "credit_card",
    "credit card": "credit_card",
    "prepaid card": "credit_card",
    # cuentas
    "checking or savings account": "bank_account",
    "bank account or service": "bank_account",
    # préstamo estudiantil
    "student loan": "student_loan",
    # préstamos al consumo
    "vehicle loan or lease": "consumer_loan",
    "consumer loan": "consumer_loan",
    "payday loan, title loan, personal loan, or advance loan": "consumer_loan",
    "payday loan, title loan, or personal loan": "consumer_loan",
    "payday loan": "consumer_loan",
    # transferencias / dinero virtual
    "money transfer, virtual currency, or money service": "money_transfer",
    "money transfers": "money_transfer",
    "virtual currency": "money_transfer",
}
ALIASES = {
    "date":    ["date received", "date_received", "datereceived", "date"],
    "product": ["product"],
    "text":    ["consumer complaint narrative", "consumer_complaint_narrative", "complaint_what_happened",
                "complaint what happened", "narrative"],
}

def _norm(c):
    return re.sub(r"[^a-z]", "", str(c).lower())

def norm_key(t):
    return re.sub(r"[^a-z0-9]+", "", str(t).lower())

def _csv_members(zpath):
    with zipfile.ZipFile(zpath) as z:
        return [i.filename for i in z.infolist() if i.filename.lower().endswith(".csv")]

def load_data(paths):
    """Lee los ZIP/CSV por bloques (cabe en 8 GB de RAM). Detecta los nombres de columna aunque cambien."""
    parts = []
    for path in paths:
        zf = zipfile.ZipFile(path) if path.lower().endswith(".zip") else None
        members = _csv_members(path) if zf else [None]
        for m in members:
            opener = (lambda: zf.open(m)) if zf else (lambda: path)
            head = pd.read_csv(opener(), nrows=0)
            cols = {k: next((c for c in head.columns if _norm(c) in [_norm(x) for x in al]), None) for k, al in ALIASES.items()}
            faltan = [k for k, v in cols.items() if v is None]
            if faltan:
                raise KeyError(f"{os.path.basename(path)}: no encuentro {faltan}. Columnas del archivo: {list(head.columns)}")
            n0 = sum(len(p) for p in parts)
            for ch in pd.read_csv(opener(), usecols=list(cols.values()), chunksize=400_000, low_memory=False):
                ch = ch.rename(columns={v: k for k, v in cols.items()})
                ch = ch.dropna(subset=["text", "product"])
                ch["year"] = pd.to_datetime(ch["date"], errors="coerce").dt.year
                ch = ch[(ch["year"] >= CONFIG["START_YEAR"]) & (ch["year"] <= CONFIG["END_YEAR"])]
                parts.append(ch[["year", "product", "text"]])
            print(f"  {os.path.basename(path)}: +{sum(len(p) for p in parts)-n0:,} reclamos con narrativa")
    if not parts:
        raise ValueError("No se leyó ningún CSV dentro de los ZIP.")
    out = pd.concat(parts, ignore_index=True).rename(columns={"product": "product_raw"})
    if len(out) == 0:
        raise ValueError(f"Ningún reclamo con narrativa entre {CONFIG['START_YEAR']} y {CONFIG['END_YEAR']}.")
    return out

df = load_data(DATA_FILES)
print("Registros con narrativa en el periodo:", len(df))
print(df["product_raw"].value_counts())

  CCDB_Export_2_May_2018_through_April_2021.zip: +240,398 reclamos con narrativa
  CCDB_Export_3_May_2021_through_October_2022.zip: +408,046 reclamos con narrativa
  CCDB_Export_4_November_2022_through_August_2023.zip: +388,495 reclamos con narrativa
  CCDB_Export_5_September_2023_through_March_2024.zip: +318,804 reclamos con narrativa
  CCDB_Export_6_April_2024_through_July_2024.zip: +276,332 reclamos con narrativa
  CCDB_Export_7_August_2024_through_October_2024.zip: +229,944 reclamos con narrativa
  CCDB_Export_8_November_2024_through_December_2024.zip: +154,916 reclamos con narrativa
  CCDB_Export_9_January_2025_through_February_2025.zip: +265,405 reclamos con narrativa
  CCDB_Export_10_March_2025_through_April_2025.zip: +210,147 reclamos con narrativa
  CCDB_Export_11_May_2025_through_June_2025.zip: +219,632 reclamos con narrativa
  CCDB_Export_12_July_2025_through_August_2025.zip: +234,404 reclamos con narrativa
  CCDB_Export_13_September_2025_through_October_2025.zip: +157,821 r

In [6]:
df["label"] = df["product_raw"].str.lower().str.strip().map(PRODUCT_MAP)
sin_mapa = df[df["label"].isna()]
print(f"Productos sin mapear: {sin_mapa['product_raw'].nunique()} tipos, {len(sin_mapa)} registros descartados")
print(sin_mapa["product_raw"].value_counts().head(10))
df = df.dropna(subset=["label"]).copy()

# depuración: texto mínimo y duplicados exactos (tras normalizar) para evitar fuga entre particiones
df["n_words"] = df["text"].str.split().str.len()
df = df[df["n_words"] >= CONFIG["MIN_WORDS"]]
df["dup_key"] = df["text"].map(lambda t: hashlib.md5(re.sub(r"[^a-z0-9]", "", t.lower()).encode()).hexdigest())
antes = len(df)
df = df.drop_duplicates("dup_key")
print(f"Duplicados eliminados: {antes - len(df)}")

# muestra de trabajo (distribución natural de clases)
if len(df) > CONFIG["N_SAMPLE"]:
    df = df.sample(CONFIG["N_SAMPLE"], random_state=42)
df = df.reset_index(drop=True)
CLASSES = sorted(df["label"].unique())
CLASS2ID = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(CLASS2ID)
print("Clases:", CLASSES, "| N =", len(df))

# --- Tabla 7 (EDA): distribución de clases y longitud
eda = (df.groupby("label")
         .agg(n=("y", "size"), palabras_media=("n_words", "mean"), palabras_mediana=("n_words", "median"))
         .assign(porcentaje=lambda d: (100 * d["n"] / len(df)).round(2))
         .sort_values("n", ascending=False).round(1))
eda.to_csv(f"{CONFIG['OUT_DIR']}/tabla7_distribucion_clases.csv")
print(eda)
print("Longitud global (palabras): media %.1f | mediana %.0f | p95 %.0f | máx %d" % (
    df.n_words.mean(), df.n_words.median(), df.n_words.quantile(.95), df.n_words.max()))
df["n_words"].clip(upper=1500).hist(bins=60); plt.xlabel("Palabras por reclamo (recortado a 1500)")
plt.ylabel("Frecuencia"); plt.tight_layout(); plt.savefig(f"{CONFIG['OUT_DIR']}/fig_longitud.png", dpi=200); plt.close()

Productos sin mapear: 1 tipos, 4505 registros descartados
product_raw
Debt or credit management    4505
Name: count, dtype: int64
Duplicados eliminados: 1314277
Clases: ['bank_account', 'consumer_loan', 'credit_card', 'credit_reporting', 'debt_collection', 'money_transfer', 'mortgage', 'student_loan'] | N = 60000
                      n  palabras_media  palabras_mediana  porcentaje
label                                                                
credit_reporting  34955           209.4             141.0        58.3
debt_collection    7478           194.8             124.0        12.5
credit_card        5098           216.2             162.0         8.5
bank_account       4679           218.3             155.0         7.8
mortgage           2414           296.9             213.5         4.0
money_transfer     2337           186.1             118.0         3.9
consumer_loan      2005           225.7             162.0         3.3
student_loan       1034           224.6             179

## 2. Preprocesamiento (dos tuberías)
* **Clásica** (Naive Bayes y SVM): minúsculas, eliminación de marcas `XXXX`, de números y signos,
  palabras vacías y lematización (NLTK). Después TF-IDF con unigramas y bigramas.
* **Transformer** (RoBERTa): texto casi original (solo se normalizan las marcas de anonimización y los
  espacios), porque el tokenizador de subpalabras aprovecha las palabras vacías y la morfología.

In [7]:
try:
    import nltk
    for r in ["wordnet", "omw-1.4", "stopwords"]:
        nltk.download(r, quiet=True)
    from nltk.stem import WordNetLemmatizer
    from nltk.corpus import stopwords
    _lem = WordNetLemmatizer(); _lem.lemmatize("tests")
    STOP = set(stopwords.words("english"))
    LEMMA_OK = True
except Exception as e:
    print("AVISO: no se pudo cargar NLTK/WordNet (", e, ") -> se usa lista de stopwords de sklearn y SIN lematización.")
    _lem = None; STOP = set(ENGLISH_STOP_WORDS); LEMMA_OK = False

from functools import lru_cache
@lru_cache(maxsize=None)
def _lemma(w):
    return _lem.lemmatize(w) if _lem is not None else w

RE_X = re.compile(r"\bx{2,}\b", re.I)
RE_NONALPHA = re.compile(r"[^a-z\s]")
def prep_classic(t):
    t = RE_X.sub(" ", t.lower())
    t = RE_NONALPHA.sub(" ", t)
    return " ".join(_lemma(w) for w in t.split() if w not in STOP and len(w) > 1)

RE_XX = re.compile(r"(?:\{?\$?[\d.,]+\}?\s*)?\b[Xx]{2,}\b(?:\s+\b[Xx]{2,}\b)*")
def prep_transformer(t):
    t = re.sub(r"\b[Xx]{2,}\b(?:\s+\b[Xx]{2,}\b)+", "XXXX", t)   # colapsa XXXX XXXX -> XXXX
    return re.sub(r"\s+", " ", t).strip()

t0 = time.time()
df["text_classic"] = df["text"].map(prep_classic)
df["text_tf"] = df["text"].map(prep_transformer)
print(f"Preprocesamiento listo en {time.time()-t0:.0f} s | lematización: {LEMMA_OK}")
print("Ejemplo original :", df.text.iloc[0][:200])
print("Ejemplo clásico  :", df.text_classic.iloc[0][:200])

Preprocesamiento listo en 12 s | lematización: True
Ejemplo original : Last year, I discovered an error in my XXXX credit report regarding my address. I promptly disputed it online and reached out to XXXX  call center. They informed me that Rocket Mortgage had mistakenly
Ejemplo clásico  : last year discovered error credit report regarding address promptly disputed online reached call center informed rocket mortgage mistakenly reported incorrect street address unit number contacted rock


## 3. Métricas, coste y utilidades de evaluación

In [8]:
LABELS = list(range(len(CLASSES)))

def metrics(y, p):
    pm, rm, fm, _ = precision_recall_fscore_support(y, p, labels=LABELS, average="macro", zero_division=0)
    return dict(accuracy=accuracy_score(y, p), precision_macro=pm, recall_macro=rm, f1_macro=fm,
                f1_weighted=f1_score(y, p, labels=LABELS, average="weighted", zero_division=0))

def per_class_f1(y, p):
    return f1_score(y, p, labels=LABELS, average=None, zero_division=0)

def bootstrap_ci(y, p, n=1000, seed=0):
    rng = np.random.RandomState(seed); y = np.asarray(y); p = np.asarray(p); out = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        out.append(f1_score(y[i], p[i], labels=LABELS, average="macro", zero_division=0))
    return np.percentile(out, [2.5, 97.5])

def mcnemar_p(y, pa, pb):
    """Prueba de McNemar con corrección de continuidad (Dietterich, 1998)."""
    from scipy.stats import chi2
    a = (np.asarray(pa) == np.asarray(y)); b = (np.asarray(pb) == np.asarray(y))
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    if n01 + n10 == 0:
        return 1.0
    stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    return float(chi2.sf(stat, 1))

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pvals[i]); adj[i] = min(1.0, run)
    return adj

BINS = [(0, 100, "≤100"), (101, 250, "101–250"), (251, 500, "251–500"), (501, 10**9, ">500")]

## 4. Modelos clásicos: Naive Bayes y SVM lineal (TF-IDF, ponderación de clases)

In [9]:
def run_classic(tr, va, te, seed):
    res = {}
    t0 = time.time()
    vec = TfidfVectorizer(**CONFIG["TFIDF"])
    Xtr = vec.fit_transform(tr["text_classic"]); Xva = vec.transform(va["text_classic"])
    t_vec = time.time() - t0

    # --- Naive Bayes (priors uniformes = equivalente a ponderar por clase)
    best = (-1, None)
    t0 = time.time()
    for a in CONFIG["NB_ALPHAS"]:
        m = MultinomialNB(alpha=a, fit_prior=False).fit(Xtr, tr["y"])
        f = f1_score(va["y"], m.predict(Xva), labels=LABELS, average="macro", zero_division=0)
        if f > best[0]: best = (f, a)
    nb = MultinomialNB(alpha=best[1], fit_prior=False)
    t1 = time.time(); nb.fit(Xtr, tr["y"]); t_fit_nb = time.time() - t1
    res["Naive Bayes"] = dict(model=nb, hp={"alpha": best[1]}, val_f1=best[0],
                              train_s=t_vec + t_fit_nb, tune_s=time.time() - t0)
    # --- SVM lineal
    best = (-1, None)
    t0 = time.time()
    for C in CONFIG["SVM_CS"]:
        m = LinearSVC(C=C, class_weight="balanced", max_iter=5000, random_state=seed).fit(Xtr, tr["y"])
        f = f1_score(va["y"], m.predict(Xva), labels=LABELS, average="macro", zero_division=0)
        if f > best[0]: best = (f, C)
    svm = LinearSVC(C=best[1], class_weight="balanced", max_iter=5000, random_state=seed)
    t1 = time.time(); svm.fit(Xtr, tr["y"]); t_fit_svm = time.time() - t1
    res["SVM lineal"] = dict(model=svm, hp={"C": best[1]}, val_f1=best[0],
                             train_s=t_vec + t_fit_svm, tune_s=time.time() - t0)

    # --- predicción sobre prueba + latencia (incluye vectorización; texto ya preprocesado)
    out = {}
    for name, r in res.items():
        t0 = time.time()
        Xte = vec.transform(te["text_classic"]); pred = r["model"].predict(Xte)
        lat = 1000 * (time.time() - t0) / len(te)
        out[name] = dict(pred=pred, hp=r["hp"], val_f1=r["val_f1"], train_s=r["train_s"],
                         infer_ms_cpu=lat, infer_ms_gpu=np.nan, n_params=int(np.prod(Xtr.shape[1:]) * len(CLASSES)),
                         gpu_mem_gb=np.nan, vec=vec, model=r["model"])
    return out

## 5. Transformer: RoBERTa-base con fine-tuning (pérdida ponderada, truncamiento inicio+final)

In [10]:
def run_transformer(tr, va, te, seed):
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if dev.type == "cpu":
        print("AVISO: no hay GPU; el entrenamiento será muy lento. Activa GPU en Colab.")
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
    L, H = CONFIG["MAX_LEN"], CONFIG["HEAD_TOKENS"]
    body = L - 2   # <s> y </s>

    def encode(texts):
        ids = tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
        out = []
        for x in ids:
            if len(x) > body:                       # inicio + final (Sun et al., 2019)
                x = x[:H] + x[-(body - H):]
            out.append([tok.cls_token_id] + x + [tok.sep_token_id])   # <s> ... </s> (RoBERTa/BERT)
        return out

    def batches(enc, y=None, bs=32, shuffle=False):
        idx = np.arange(len(enc))
        if shuffle: np.random.shuffle(idx)
        else: idx = np.argsort([len(e) for e in enc])   # inferencia: ordena por longitud
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]; m = max(len(enc[j]) for j in b)
            ids = torch.full((len(b), m), tok.pad_token_id, dtype=torch.long); att = torch.zeros((len(b), m), dtype=torch.long)
            for k, j in enumerate(b):
                ids[k, :len(enc[j])] = torch.tensor(enc[j]); att[k, :len(enc[j])] = 1
            yield b, ids, att, (torch.tensor(y[b]) if y is not None else None)

    enc_tr, enc_va, enc_te = encode(tr["text_tf"]), encode(va["text_tf"]), encode(te["text_tf"])
    ytr, yva = tr["y"].values, va["y"].values

    model = AutoModelForSequenceClassification.from_pretrained(CONFIG["MODEL_NAME"], num_labels=len(CLASSES)).to(dev)
    cnt = np.bincount(ytr, minlength=len(CLASSES)).astype(float)
    w = torch.tensor(cnt.sum() / (len(CLASSES) * np.maximum(cnt, 1)), dtype=torch.float, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w)
    nd = ["bias", "LayerNorm.weight"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nd)], "weight_decay": CONFIG["WEIGHT_DECAY"]},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=CONFIG["LR"])
    steps = CONFIG["EPOCHS"] * int(np.ceil(len(enc_tr) / CONFIG["BATCH"]))
    sch = get_linear_schedule_with_warmup(opt, int(CONFIG["WARMUP"] * steps), steps)
    use_amp = dev.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    @torch.no_grad()
    def predict(enc, bs=64):
        model.eval(); pred = np.zeros(len(enc), dtype=int)
        for b, ids, att, _ in batches(enc, bs=bs):
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                lo = model(input_ids=ids.to(dev), attention_mask=att.to(dev)).logits
            pred[b] = lo.argmax(-1).cpu().numpy()
        return pred

    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    best_f1, best_state, t_train = -1, None, 0.0
    for ep in range(CONFIG["EPOCHS"]):
        model.train(); t0 = time.time()
        for b, ids, att, yb in batches(enc_tr, ytr, CONFIG["BATCH"], shuffle=True):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                lo = model(input_ids=ids.to(dev), attention_mask=att.to(dev)).logits
            loss = lossf(lo.float(), yb.to(dev))
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        t_train += time.time() - t0
        f = f1_score(yva, predict(enc_va), labels=LABELS, average="macro", zero_division=0)
        print(f"  [seed {seed}] época {ep+1}/{CONFIG['EPOCHS']} · F1 macro val = {f:.4f} · {time.time()-t0:.0f} s")
        if f > best_f1:
            best_f1 = f; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    mem = torch.cuda.max_memory_allocated() / 1e9 if dev.type == "cuda" else np.nan

    # latencia de inferencia en GPU (lote 64) sobre todo el conjunto de prueba
    if dev.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); pred = predict(enc_te)
    if dev.type == "cuda": torch.cuda.synchronize()
    ms_gpu = 1000 * (time.time() - t0) / len(enc_te)

    # latencia en CPU (subconjunto) para reflejar un despliegue sin GPU
    n = min(CONFIG["CPU_LATENCY_N"], len(enc_te)); sub = enc_te[:n]
    model.to("cpu"); torch.set_num_threads(os.cpu_count() or 2); use_amp_old = use_amp; use_amp = False
    dev_old, dev = dev, torch.device("cpu")
    t0 = time.time(); _ = predict(sub, bs=16); ms_cpu = 1000 * (time.time() - t0) / n
    dev, use_amp = dev_old, use_amp_old
    n_params = sum(p.numel() for p in model.parameters())
    del model, opt;
    if dev.type == "cuda": torch.cuda.empty_cache()
    return {"RoBERTa-base": dict(pred=pred, hp={"lr": CONFIG["LR"], "epochs": CONFIG["EPOCHS"], "batch": CONFIG["BATCH"]},
                                 val_f1=best_f1, train_s=t_train, infer_ms_cpu=ms_cpu, infer_ms_gpu=ms_gpu,
                                 n_params=n_params, gpu_mem_gb=mem)}

## 6. Experimento principal: tres semillas (partición estratificada distinta en cada una)

In [11]:
def split_df(df, seed):
    a, b, c = CONFIG["SPLIT"]
    tr, tmp = train_test_split(df, test_size=1 - a, stratify=df["y"], random_state=seed)
    va, te = train_test_split(tmp, test_size=c / (b + c), stratify=tmp["y"], random_state=seed)
    return tr, va, te

def run_all(tr, va, te, seed):
    set_seed(seed)
    out = run_classic(tr, va, te, seed)
    if not CONFIG["SKIP_TRANSFORMER"]:
        out.update(run_transformer(tr, va, te, seed))
    return out

ALL = {}            # ALL[seed][modelo] = dict(...)
TESTS = {}          # TESTS[seed] = DataFrame de test
for s in CONFIG["SEEDS"]:
    print(f"\n===== SEMILLA {s} =====")
    tr, va, te = split_df(df, s)
    print(f"train {len(tr)} · val {len(va)} · test {len(te)}")
    ALL[s] = run_all(tr, va, te, s); TESTS[s] = te.reset_index(drop=True)
    for m, r in ALL[s].items():
        mt = metrics(te["y"], r["pred"])
        print(f"  {m:14s} F1 macro={mt['f1_macro']:.4f}  acc={mt['accuracy']:.4f}  entrena={r['train_s']:.0f}s  hp={r['hp']}")


===== SEMILLA 42 =====
train 41999 · val 6000 · test 12001


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (10 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1858 > 512). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [seed 44] época 1/3 · F1 macro val = 0.7548 · 568 s
  [seed 44] época 2/3 · F1 macro val = 0.7584 · 567 s
  [seed 44] época 3/3 · F1 macro val = 0.7677 · 568 s
  Naive Bayes    F1 macro=0.6958  acc=0.7839  entrena=11s  hp={'alpha': 0.01}
  SVM lineal     F1 macro=0.7784  acc=0.8471  entrena=14s  hp={'C': 0.3}
  RoBERTa-base   F1 macro=0.7697  acc=0.8268  entrena=1658s  hp={'lr': 2e-05, 'epochs': 3, 'batch': 32}


## 7. Tablas de resultados (se guardan en CSV y Markdown)

In [12]:
MODELS = list(ALL[CONFIG["SEEDS"][0]].keys())
rows = []
for s in CONFIG["SEEDS"]:
    y = TESTS[s]["y"].values
    for m in MODELS:
        rows.append(dict(seed=s, model=m, **metrics(y, ALL[s][m]["pred"]),
                         train_s=ALL[s][m]["train_s"], infer_ms_cpu=ALL[s][m]["infer_ms_cpu"],
                         infer_ms_gpu=ALL[s][m]["infer_ms_gpu"], gpu_mem_gb=ALL[s][m]["gpu_mem_gb"],
                         n_params=ALL[s][m]["n_params"], val_f1=ALL[s][m]["val_f1"], hp=json.dumps(ALL[s][m]["hp"])))
R = pd.DataFrame(rows); R.to_csv(f"{CONFIG['OUT_DIR']}/resultados_por_semilla.csv", index=False)

def ms(x): return f"{np.mean(x):.4f} ± {np.std(x, ddof=1) if len(x) > 1 else 0:.4f}"
s0 = CONFIG["SEEDS"][0]
t8 = []
for m in MODELS:
    d = R[R.model == m]; lo, hi = bootstrap_ci(TESTS[s0]["y"].values, ALL[s0][m]["pred"])
    t8.append({"Modelo": m, "Exactitud": ms(d.accuracy), "Precisión macro": ms(d.precision_macro),
               "Exhaustividad macro": ms(d.recall_macro), "F1 macro": ms(d.f1_macro),
               f"IC 95 % F1 macro (semilla {s0})": f"[{lo:.4f}; {hi:.4f}]", "F1 ponderado": ms(d.f1_weighted)})
T8 = pd.DataFrame(t8); T8.to_csv(f"{CONFIG['OUT_DIR']}/tabla8_resultados_globales.csv", index=False)

t9 = pd.DataFrame({m: np.mean([per_class_f1(TESTS[s]["y"].values, ALL[s][m]["pred"]) for s in CONFIG["SEEDS"]], axis=0)
                   for m in MODELS}, index=CLASSES).round(4)
t9["n_prueba (semilla %d)" % s0] = np.bincount(TESTS[s0]["y"].values, minlength=len(CLASSES))
t9.to_csv(f"{CONFIG['OUT_DIR']}/tabla9_f1_por_clase.csv")

t10 = []
for m in MODELS:
    d = R[R.model == m]
    t10.append({"Modelo": m, "Parámetros": f"{int(d.n_params.iloc[0]):,}", "Entrenamiento (s)": f"{d.train_s.mean():.0f}",
                "Inferencia CPU (ms/reclamo)": f"{d.infer_ms_cpu.mean():.2f}",
                "Inferencia GPU (ms/reclamo)": "—" if d.infer_ms_gpu.isna().all() else f"{d.infer_ms_gpu.mean():.2f}",
                "Memoria GPU pico (GB)": "—" if d.gpu_mem_gb.isna().all() else f"{d.gpu_mem_gb.mean():.1f}"})
T10 = pd.DataFrame(t10); T10.to_csv(f"{CONFIG['OUT_DIR']}/tabla10_coste.csv", index=False)

# Tabla 11: McNemar por pares con corrección de Holm
pairs = [(a, b) for i, a in enumerate(MODELS) for b in MODELS[i + 1:]]
t11 = []
for s in CONFIG["SEEDS"]:
    y = TESTS[s]["y"].values
    p = [mcnemar_p(y, ALL[s][a]["pred"], ALL[s][b]["pred"]) for a, b in pairs]
    ph = holm(np.array(p))
    for (a, b), pr, pa in zip(pairs, p, ph):
        t11.append({"semilla": s, "comparación": f"{a} vs {b}", "p (McNemar)": pr, "p ajustado (Holm)": pa, "significativo (α=0,05)": pa < 0.05})
T11 = pd.DataFrame(t11); T11.to_csv(f"{CONFIG['OUT_DIR']}/tabla11_mcnemar.csv", index=False)

# Tabla 12: F1 macro por tramo de longitud (palabras del texto original)
t12 = []
for lo, hi, name in BINS:
    row = {"Tramo (palabras)": name}
    for m in MODELS:
        vals = []
        for s in CONFIG["SEEDS"]:
            te = TESTS[s]; mask = (te["n_words"] >= lo) & (te["n_words"] <= hi)
            if mask.sum() > 0:
                yy = te.loc[mask, "y"].values; pp = ALL[s][m]["pred"][mask.values]
                vals.append(f1_score(yy, pp, labels=np.unique(yy), average="macro", zero_division=0))
        row[m] = round(float(np.mean(vals)), 4) if vals else np.nan
    row["n (semilla %d)" % s0] = int(((TESTS[s0]["n_words"] >= lo) & (TESTS[s0]["n_words"] <= hi)).sum())
    t12.append(row)
T12 = pd.DataFrame(t12); T12.to_csv(f"{CONFIG['OUT_DIR']}/tabla12_f1_por_longitud.csv", index=False)

for name, T in [("Tabla 7 · Distribución de clases", eda.reset_index()), ("Tabla 8 · Resultados globales", T8),
                ("Tabla 9 · F1 por clase", t9.reset_index().rename(columns={"index": "clase"})), ("Tabla 10 · Coste computacional", T10),
                ("Tabla 11 · McNemar", T11), ("Tabla 12 · F1 macro por longitud", T12)]:
    print("\n" + name); print(T.to_string(index=False))


Tabla 7 · Distribución de clases
           label     n  palabras_media  palabras_mediana  porcentaje
credit_reporting 34955           209.4             141.0        58.3
 debt_collection  7478           194.8             124.0        12.5
     credit_card  5098           216.2             162.0         8.5
    bank_account  4679           218.3             155.0         7.8
        mortgage  2414           296.9             213.5         4.0
  money_transfer  2337           186.1             118.0         3.9
   consumer_loan  2005           225.7             162.0         3.3
    student_loan  1034           224.6             179.0         1.7

Tabla 8 · Resultados globales
      Modelo       Exactitud Precisión macro Exhaustividad macro        F1 macro IC 95 % F1 macro (semilla 42)    F1 ponderado
 Naive Bayes 0.7877 ± 0.0042 0.6720 ± 0.0049     0.7559 ± 0.0081 0.7009 ± 0.0044              [0.6907; 0.7143] 0.7950 ± 0.0040
  SVM lineal 0.8495 ± 0.0043 0.7649 ± 0.0064     0.7988 ± 0.

## 8. Figuras (matrices de confusión, F1 por clase)

In [13]:
for m in MODELS:
    cm = confusion_matrix(TESTS[s0]["y"], ALL[s0][m]["pred"], labels=LABELS, normalize="true")
    fig, ax = plt.subplots(figsize=(6.8, 5.8)); im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > .5 else "black", fontsize=7)
    ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title(f"{m} · matriz de confusión normalizada (semilla {s0})", fontsize=9)
    fig.tight_layout(); fig.savefig(f"{CONFIG['OUT_DIR']}/cm_{m.replace(' ', '_')}.png", dpi=200); plt.close(fig)
t9[MODELS].plot.bar(figsize=(9, 4.5)); plt.ylabel("F1 (media de semillas)"); plt.ylim(0, 1); plt.tight_layout()
plt.savefig(f"{CONFIG['OUT_DIR']}/fig_f1_por_clase.png", dpi=200); plt.close()

# informe por clase (precisión/recall/F1) de la semilla principal
for m in MODELS:
    rep = classification_report(TESTS[s0]["y"], ALL[s0][m]["pred"], labels=LABELS, target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(rep).T.round(4).to_csv(f"{CONFIG['OUT_DIR']}/informe_clases_{m.replace(' ', '_')}.csv")

## 9. Experimento complementario: partición temporal (deriva)
Se entrena con los años anteriores al último y se evalúa con el último año completo del periodo.

In [14]:
T13 = None
if CONFIG["RUN_TEMPORAL"] and df["year"].nunique() > 1:
    last = int(df["year"].max())
    old, new = df[df["year"] < last], df[df["year"] == last]
    tr_t, va_t = train_test_split(old, test_size=0.11, stratify=old["y"], random_state=42)
    print(f"Temporal: train {len(tr_t)} (años < {last}) · val {len(va_t)} · test {len(new)} (año {last})")
    out_t = run_all(tr_t, va_t, new, 42)
    T13 = pd.DataFrame([{"Modelo": m, "F1 macro (temporal)": round(metrics(new['y'], r['pred'])['f1_macro'], 4),
                         "Exactitud (temporal)": round(metrics(new['y'], r['pred'])['accuracy'], 4)} for m, r in out_t.items()])
    T13 = T13.merge(R[R.seed == s0][["model", "f1_macro"]].rename(columns={"model": "Modelo", "f1_macro": "F1 macro (aleatoria, semilla 42)"}).round(4), on="Modelo")
    T13.to_csv(f"{CONFIG['OUT_DIR']}/tabla13_temporal.csv", index=False); print(T13)

Temporal: train 37000 (años < 2025) · val 4574 · test 18426 (año 2025)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (730 > 512). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [seed 42] época 1/3 · F1 macro val = 0.7485 · 500 s
  [seed 42] época 2/3 · F1 macro val = 0.7669 · 498 s
  [seed 42] época 3/3 · F1 macro val = 0.7763 · 499 s
         Modelo  F1 macro (temporal)  Exactitud (temporal)  \
0   Naive Bayes               0.7018                0.7993   
1    SVM lineal               0.7665                0.8429   
2  RoBERTa-base               0.7744                0.8357   

   F1 macro (aleatoria, semilla 42)  
0                            0.7032  
1                            0.7886  
2                            0.7688  


## 10. Entorno y exportación a Markdown (para pegar en el Word)

In [15]:
env = dict(python=platform.python_version(), platform=platform.platform(), config={k: v for k, v in CONFIG.items()}, lemmatization=LEMMA_OK)
try:
    import sklearn, torch, transformers
    env.update(sklearn=sklearn.__version__, torch=torch.__version__, transformers=transformers.__version__,
               gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna"))
except Exception:
    import sklearn; env.update(sklearn=sklearn.__version__)
env["n_total"] = int(len(df)); env["clases"] = CLASSES
json.dump(env, open(f"{CONFIG['OUT_DIR']}/entorno.json", "w"), indent=2, default=str)

def to_md(T):
    try:
        return T.to_markdown(index=False)
    except Exception:
        return "```\n" + T.to_string(index=False) + "\n```"

with open(f"{CONFIG['OUT_DIR']}/tablas_capitulo5.md", "w", encoding="utf-8") as f:
    for name, T in [("Tabla 7. Distribución de clases y longitud", eda.reset_index()), ("Tabla 8. Resultados globales", T8),
                    ("Tabla 9. F1 por clase", t9.reset_index().rename(columns={"index": "clase"})), ("Tabla 10. Coste computacional", T10),
                    ("Tabla 11. McNemar con Holm", T11), ("Tabla 12. F1 macro por longitud", T12)] + ([("Tabla 13. Partición temporal", T13)] if T13 is not None else []):
        f.write(f"### {name}\n\n{to_md(T)}\n\n")
print("Listo. Descarga la carpeta:", CONFIG["OUT_DIR"])

Listo. Descarga la carpeta: resultados


In [16]:
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h | head -2

CPU(s):                                  8
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
               total        used        free      shared  buff/cache   available
Mem:            50Gi        13Gi        30Gi        16Mi       7.3Gi        37Gi


In [17]:
raw = load_data(DATA_FILES)
raw["label"] = raw["product_raw"].str.lower().str.strip().map(PRODUCT_MAP)
raw = raw.dropna(subset=["label"])
nw = raw["text"].str.split().str.len()
print("Con clase asignada:", len(raw))
print("Menos de 10 palabras:", int((nw < CONFIG["MIN_WORDS"]).sum()))
print("Quedan tras el filtro:", int((nw >= CONFIG["MIN_WORDS"]).sum()))
del raw, nw

  CCDB_Export_2_May_2018_through_April_2021.zip: +240,398 reclamos con narrativa
  CCDB_Export_3_May_2021_through_October_2022.zip: +408,046 reclamos con narrativa
  CCDB_Export_4_November_2022_through_August_2023.zip: +388,495 reclamos con narrativa
  CCDB_Export_5_September_2023_through_March_2024.zip: +318,804 reclamos con narrativa
  CCDB_Export_6_April_2024_through_July_2024.zip: +276,332 reclamos con narrativa
  CCDB_Export_7_August_2024_through_October_2024.zip: +229,944 reclamos con narrativa
  CCDB_Export_8_November_2024_through_December_2024.zip: +154,916 reclamos con narrativa
  CCDB_Export_9_January_2025_through_February_2025.zip: +265,405 reclamos con narrativa
  CCDB_Export_10_March_2025_through_April_2025.zip: +210,147 reclamos con narrativa
  CCDB_Export_11_May_2025_through_June_2025.zip: +219,632 reclamos con narrativa
  CCDB_Export_12_July_2025_through_August_2025.zip: +234,404 reclamos con narrativa
  CCDB_Export_13_September_2025_through_October_2025.zip: +157,821 r

In [18]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
n_tok = [len(x) for x in tok(list(df["text_tf"]), add_special_tokens=False)["input_ids"]]
n_tok = np.array(n_tok)
print("% que supera 254 tokens:", round(100*(n_tok > 254).mean(), 1))
print("Mediana de tokens:", int(np.median(n_tok)), "| p95:", int(np.percentile(n_tok, 95)))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1321 > 512). Running this sequence through the model will result in indexing errors


% que supera 254 tokens: 32.1
Mediana de tokens: 167 | p95: 710
